# J9b : LightGBM + Optuna (Anti-Overfitting Pipeline)

**Features:** Mean-pooled MoCo (2048d)

**Anti-Overfitting:**
- Search space conservateur Optuna
- Patient-level StratifiedKFold CV
- Early stopping
- Sauvegarde OOF + Test preds pour stacking

**Stacking:** La dernière section combine les prédictions XGB + LGBM
pour une soumission fusionnée.


In [ ]:
# ============================================
# 1. MOUNT DRIVE & CONFIG
# ============================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA = '/content/drive/MyDrive/OWKIN_ML2/Data'
DRIVE_OUTPUT = '/content/drive/MyDrive/OWKIN_ML2/J9_outputs'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

TRAIN_MOCO = os.path.join(DRIVE_DATA, 'train_input', 'moco_features')
TEST_MOCO  = os.path.join(DRIVE_DATA, 'test_input', 'moco_features')
TRAIN_LABELS = os.path.join(DRIVE_DATA, 'train_output.csv')
META_TRAIN = os.path.join(DRIVE_DATA, 'supplementary_data', 'train_metadata.csv')
META_TEST  = os.path.join(DRIVE_DATA, 'supplementary_data', 'test_metadata.csv')

for p in [TRAIN_MOCO, TEST_MOCO, TRAIN_LABELS, META_TRAIN]:
    assert os.path.exists(p), f'NOT FOUND: {p}'
print('All paths OK ✅')


In [ ]:
import numpy as np
import pandas as pd
import json, shutil
from tqdm.notebook import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import lightgbm as lgb
print('Imports OK ✅')


In [ ]:
# ============================================
# 2. DATA LOADING : Mean-Pool MoCo Features
# ============================================
df_meta_train = pd.read_csv(META_TRAIN)
df_meta_test  = pd.read_csv(META_TEST)
df_labels = pd.read_csv(TRAIN_LABELS)
df_train = df_meta_train.merge(df_labels, on='Sample ID')

def load_mean_features(sample_ids, moco_dir):
    features = []
    for sid in tqdm(sample_ids, desc='Loading'):
        data = np.load(os.path.join(moco_dir, sid))
        feats = data[:, 3:]  # drop (x, y, zoom)
        features.append(np.mean(feats, axis=0))  # Mean pool → 2048d
    return np.array(features)

X_train = load_mean_features(df_train['Sample ID'].values, TRAIN_MOCO)
X_test  = load_mean_features(df_meta_test['Sample ID'].values, TEST_MOCO)
y_train = df_train['Target'].values
patients_train = df_train['Patient ID'].values

# Patient-level CV setup
patients_unique = np.unique(patients_train)
y_patient = np.array([int(np.mean(y_train[patients_train == p])) for p in patients_unique])

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Patients uniques: {len(patients_unique)} | Positive rate: {y_train.mean():.2%}')


In [ ]:
# ============================================
# 3. OPTUNA : Conservative LightGBM Tuning
# ============================================
N_FOLDS = 5
N_OPTUNA_TRIALS = 40

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'device': 'gpu',
        'verbose': -1,
        'seed': 42,
        # ====== CONSERVATIVE SEARCH SPACE ======
        'max_depth':          trial.suggest_int('max_depth', 2, 5),         # SHALLOW
        'num_leaves':         trial.suggest_int('num_leaves', 8, 31),       # LOW
        'learning_rate':      trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'n_estimators':       trial.suggest_int('n_estimators', 100, 600),
        'min_child_samples':  trial.suggest_int('min_child_samples', 10, 50),  # HIGH → anti-overfit
        'subsample':          trial.suggest_float('subsample', 0.5, 0.8),
        'colsample_bytree':   trial.suggest_float('colsample_bytree', 0.1, 0.4),  # LOW
        'reg_alpha':          trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda':         trial.suggest_float('reg_lambda', 1.0, 50.0, log=True),
        'min_split_gain':     trial.suggest_float('min_split_gain', 0.01, 1.0, log=True),
    }
    
    aucs = []
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=42)
    
    for train_idx, val_idx in skf.split(patients_unique, y_patient):
        train_mask = np.isin(patients_train, patients_unique[train_idx])
        val_mask   = np.isin(patients_train, patients_unique[val_idx])
        
        X_tr, y_tr = X_train[train_mask], y_train[train_mask]
        X_val, y_val = X_train[val_mask], y_train[val_mask]
        
        dtrain = lgb.Dataset(X_tr, y_tr)
        dval   = lgb.Dataset(X_val, y_val, reference=dtrain)
        
        model = lgb.train(
            params, dtrain,
            num_boost_round=params['n_estimators'],
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
        )
        preds = model.predict(X_val)
        aucs.append(roc_auc_score(y_val, preds))
    
    return np.mean(aucs)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)

print(f'\n✅ Best CV AUC: {study.best_value:.4f}')
print(f'Best params: {study.best_params}')


In [ ]:
# ============================================
# 4. FINAL TRAINING : OOF + Test Predictions
# ============================================
best_params = study.best_params.copy()
best_params.update({
    'objective': 'binary',
    'metric': 'auc',
    'device': 'gpu',
    'verbose': -1,
})
n_est = best_params.pop('n_estimators')

N_REPEATS = 3  # 3 repeats × 5 folds = 15 models
oof_preds = np.zeros(len(X_train))
oof_counts = np.zeros(len(X_train))
test_preds = np.zeros(len(X_test))
all_aucs = []

for repeat in range(N_REPEATS):
    skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=repeat)
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(patients_unique, y_patient)):
        train_mask = np.isin(patients_train, patients_unique[train_idx])
        val_mask   = np.isin(patients_train, patients_unique[val_idx])
        
        X_tr, y_tr = X_train[train_mask], y_train[train_mask]
        X_val, y_val = X_train[val_mask], y_train[val_mask]
        
        dtrain = lgb.Dataset(X_tr, y_tr)
        dval   = lgb.Dataset(X_val, y_val, reference=dtrain)
        
        bp = best_params.copy()
        bp['seed'] = 42 + repeat * 100 + fold
        
        model = lgb.train(
            bp, dtrain,
            num_boost_round=n_est,
            valid_sets=[dval],
            callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
        )
        
        val_preds = model.predict(X_val)
        auc = roc_auc_score(y_val, val_preds)
        all_aucs.append(auc)
        
        oof_preds[val_mask] += val_preds
        oof_counts[val_mask] += 1
        test_preds += model.predict(X_test)
        
        print(f'  Repeat {repeat+1} Fold {fold+1}: AUC = {auc:.4f}')

# Average
oof_preds /= np.maximum(oof_counts, 1)
test_preds /= (N_REPEATS * N_FOLDS)

overall_oof_auc = roc_auc_score(y_train, oof_preds)
print(f'\n{"="*50}')
print(f'LightGBM Overall OOF AUC: {overall_oof_auc:.4f}')
print(f'Mean fold AUC: {np.mean(all_aucs):.4f} ± {np.std(all_aucs):.4f}')
print(f'Test pred mean: {test_preds.mean():.4f} | std: {test_preds.std():.4f}')
print(f'{"="*50}')


In [ ]:
# ============================================
# 5. SAVE LGBM OUTPUTS TO DRIVE
# ============================================
# 5a. Individual submission
sub_lgb = pd.DataFrame({
    'Sample ID': df_meta_test['Sample ID'].values,
    'Target': test_preds
}).sort_values('Sample ID')

assert sub_lgb.shape == (149, 2)
assert all(sub_lgb['Target'].between(0, 1))

sub_lgb.to_csv(os.path.join(DRIVE_OUTPUT, 'submission_lgbm.csv'), index=False)
print('✅ submission_lgbm.csv saved')

# 5b. OOF predictions (for stacking)
np.save(os.path.join(DRIVE_OUTPUT, 'oof_lgbm.npy'), oof_preds)
np.save(os.path.join(DRIVE_OUTPUT, 'test_lgbm.npy'), test_preds)
print('✅ oof_lgbm.npy + test_lgbm.npy saved (for stacking)')

# 5c. Experiment params
experiment = {
    'model': 'lightgbm',
    'best_params': best_params,
    'n_estimators': n_est,
    'best_cv_auc': float(study.best_value),
    'overall_oof_auc': float(overall_oof_auc),
    'n_folds': N_FOLDS,
    'n_repeats': N_REPEATS,
    'n_optuna_trials': N_OPTUNA_TRIALS,
    'features': 'mean_pool_2048d',
}
with open(os.path.join(DRIVE_OUTPUT, 'params_lgbm.json'), 'w') as f:
    json.dump(experiment, f, indent=4, default=str)
print('✅ params_lgbm.json saved')


---
# 🔗 STACKING : XGB + LGBM → Soumission combinée

Cette section charge les prédictions OOF de XGBoost (du notebook J9a)
et les combine avec celles de LightGBM pour une soumission finale.

**Prerequis:** Avoir exécuté le notebook J9a (XGBoost) au préalable.


In [ ]:
# ============================================
# 6. STACKING : XGB + LGBM
# ============================================
xgb_oof_path = os.path.join(DRIVE_OUTPUT, 'oof_xgb.npy')
xgb_test_path = os.path.join(DRIVE_OUTPUT, 'test_xgb.npy')

if not os.path.exists(xgb_oof_path):
    print('⚠️ XGBoost OOF predictions not found! Run J9a first.')
    print(f'   Expected: {xgb_oof_path}')
else:
    oof_xgb = np.load(xgb_oof_path)
    test_xgb = np.load(xgb_test_path)
    oof_lgbm = np.load(os.path.join(DRIVE_OUTPUT, 'oof_lgbm.npy'))
    test_lgbm = np.load(os.path.join(DRIVE_OUTPUT, 'test_lgbm.npy'))
    
    print(f'XGB  OOF AUC: {roc_auc_score(y_train, oof_xgb):.4f}')
    print(f'LGBM OOF AUC: {roc_auc_score(y_train, oof_lgbm):.4f}')
    
    # --- Method 1: Simple Average ---
    oof_avg = 0.5 * oof_xgb + 0.5 * oof_lgbm
    test_avg = 0.5 * test_xgb + 0.5 * test_lgbm
    print(f'\nSimple Average OOF AUC: {roc_auc_score(y_train, oof_avg):.4f}')
    
    # --- Method 2: Optimal Blend (grid search) ---
    best_alpha, best_auc = 0.5, 0
    for alpha in np.arange(0.1, 0.91, 0.05):
        blended = alpha * oof_xgb + (1 - alpha) * oof_lgbm
        auc = roc_auc_score(y_train, blended)
        if auc > best_auc:
            best_auc = auc
            best_alpha = alpha
    print(f'Best blend: α={best_alpha:.2f} (XGB) / {1-best_alpha:.2f} (LGBM) → OOF AUC: {best_auc:.4f}')
    
    # --- Method 3: Ridge Stacking (LogReg L2 meta-learner) ---
    X_stack_train = np.column_stack([oof_xgb, oof_lgbm])
    X_stack_test  = np.column_stack([test_xgb, test_lgbm])
    
    stack_oof = np.zeros(len(y_train))
    stack_test = np.zeros(len(X_stack_test))
    stack_aucs = []
    
    skf = StratifiedKFold(5, shuffle=True, random_state=42)
    for train_idx, val_idx in skf.split(X_stack_train, y_train):
        meta = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
        meta.fit(X_stack_train[train_idx], y_train[train_idx])
        stack_oof[val_idx] = meta.predict_proba(X_stack_train[val_idx])[:, 1]
        stack_test += meta.predict_proba(X_stack_test)[:, 1] / 5
        stack_aucs.append(roc_auc_score(y_train[val_idx], stack_oof[val_idx]))
    
    print(f'Ridge Stacking OOF AUC: {roc_auc_score(y_train, stack_oof):.4f}')
    
    # ---- Choose best method ----
    results = {
        'simple_avg': (test_avg, roc_auc_score(y_train, oof_avg)),
        f'blend_a{best_alpha:.2f}': (best_alpha * test_xgb + (1-best_alpha) * test_lgbm, best_auc),
        'ridge_stack': (stack_test, roc_auc_score(y_train, stack_oof)),
    }
    
    print(f'\n{"="*50}')
    print('STACKING RESULTS SUMMARY')
    for name, (preds, auc) in results.items():
        print(f'  {name:25s} OOF AUC = {auc:.4f}')
    print(f'{"="*50}')
    
    # Save ALL stacking submissions
    for name, (preds, auc) in results.items():
        sub = pd.DataFrame({
            'Sample ID': df_meta_test['Sample ID'].values,
            'Target': preds
        }).sort_values('Sample ID')
        sub.to_csv(os.path.join(DRIVE_OUTPUT, f'submission_stack_{name}.csv'), index=False)
    
    print(f'\n✅ 3 stacking submissions saved to {DRIVE_OUTPUT}')
    print('  - submission_stack_simple_avg.csv')
    print(f'  - submission_stack_blend_a{best_alpha:.2f}.csv')
    print('  - submission_stack_ridge_stack.csv')
